In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import numpy as np
from scipy.sparse import csr_matrix, save_npz, load_npz
import pickle
import pandas as pd


# interactions = pd.read_csv('../data/processed/interactions_clean.csv')
interactions = pd.read_csv('/content/drive/MyDrive/interactions_clean.csv')
print(f"Loaded {len(interactions)} interactions")




Loaded 25981438 interactions


In [3]:
# For explicit ratings, s_ui = rating (raw)
# For ALS, we use implicit_binary as preference
interactions['preference'] = interactions['implicit_binary']  # 0 or 1

alpha = 40
interactions['confidence'] = 1 + alpha * interactions['implicit_confidence']

print("\nPreference stats:")
print(interactions['preference'].value_counts())
print("\nConfidence stats:")
print(interactions['confidence'].describe())



Preference stats:
preference
1    16067155
0     9914283
Name: count, dtype: int64

Confidence stats:
count    2.598144e+07
mean     2.922213e+01
std      8.522409e+00
min      5.000000e+00
25%      2.500000e+01
50%      2.900000e+01
75%      3.300000e+01
max      4.100000e+01
Name: confidence, dtype: float64


In [4]:
interactions.head()

,userId,tmdb_id,explicit_value,implicit_binary,implicit_confidence,type,preference,confidence
0,38150,1600,-0.103286,1,0.8,explicit,1,33.0
1,44717,8012,-0.461146,0,0.6,explicit,0,25.0
2,44717,807,1.115038,1,1.0,explicit,1,41.0
3,44717,623,-0.701052,0,0.6,explicit,0,25.0
4,45491,8844,0.549650,1,0.8,explicit,1,33.0


In [5]:
# Load mappers
with open('/content/drive/MyDrive/user_mapper.pkl', 'rb') as f:
    user_mapper = pickle.load(f)

with open('/content/drive/MyDrive/movie_mapper.pkl', 'rb') as f:
    movie_mapper = pickle.load(f)

print(f"Users: {len(user_mapper):,}")
print(f"Movies: {len(movie_mapper):,}")


# Map indices
interactions['user_idx'] = interactions['userId'].map(user_mapper)
interactions['item_idx'] = interactions['tmdb_id'].map(movie_mapper)

# Remove missing
interactions = interactions.dropna(subset=['user_idx', 'item_idx']).copy()
interactions['user_idx'] = interactions['user_idx'].astype(int)
interactions['item_idx'] = interactions['item_idx'].astype(int)

# Build confidence
alpha = 40
interactions['confidence'] = 1 + alpha * interactions['implicit_confidence']

# Dimensions
n_users = len(user_mapper)      # 270,882
n_items = len(movie_mapper)      # 45,423

print(f"\nCorrect dimensions: {n_items} items × {n_users} users")

# Build matrix (items × users)
C = csr_matrix(
    (
        interactions['confidence'].astype(np.float32),
        (interactions['item_idx'], interactions['user_idx'])
    ),
    shape=(n_items, n_users),
    dtype=np.float32
)

print(f"Matrix shape: {C.shape}")
print(f"Non-zero entries: {C.nnz:,}")
print(f"Density: {C.nnz / (n_items * n_users) * 100:.6f}%")

# Save corrected matrix
save_npz('/content/drive/MyDrive/als_confidence_matrix.npz', C)
print("✓ Saved corrected matrix")

Users: 270,882
Movies: 45,423

Correct dimensions: 45423 items × 270882 users
Matrix shape: (45423, 270882)
Non-zero entries: 25,981,437
Density: 0.211158%
✓ Saved corrected matrix


In [6]:
!pip install implicit
!pip install mlflow

In [7]:
# Transpose matrix for implicit library

import numpy as np
import pickle
from scipy.sparse import load_npz, csr_matrix
import implicit

# Load matrix
C = load_npz('/content/drive/MyDrive/als_confidence_matrix.npz')
print(f"Original C shape: {C.shape} (items × users)")

# Implicit library expects (users × items)
C_for_training = C.T.tocsr()
print(f"Training matrix shape: {C_for_training.shape} (users × items)")

# Train model
model = implicit.als.AlternatingLeastSquares(
    factors=100,
    regularization=0.01,
    iterations=50,
    alpha=1.0,
    random_state=42,
    num_threads=0
)

print("\nTraining ALS model...")
model.fit(C_for_training)

# Save the new model
with open('/content/drive/MyDrive/als_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("✓ Saved: als_model_correct.pkl")

# Now factors will be:
# user_factors: (270882, 100) - actual user vectors
# item_factors: (45423, 100) - actual item vectors
print(f"\nCorrect model factors:")
print(f"  User factors: {model.user_factors.shape} (users × K)")
print(f"  Item factors: {model.item_factors.shape} (items × K)")

Original C shape: (45423, 270882) (items × users)
Training matrix shape: (270882, 45423) (users × items)

Training ALS model...


/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/50 [00:00<?, ?it/s]

✓ Saved: als_model_correct.pkl

Correct model factors:
  User factors: (270882, 100) (users × K)
  Item factors: (45423, 100) (items × K)


In [8]:
import mlflow
import time

start_time = time.time()
# ... training code ...
training_time = time.time() - start_time

with mlflow.start_run(run_name=f"als_k100_reg0.01_{datetime.now().strftime('%Y%m%d_%H%M')}"):

    # Parameters
    mlflow.log_param("algorithm", "ALS")
    mlflow.log_param("n_factors", 100)
    mlflow.log_param("regularization", 0.01)
    mlflow.log_param("iterations", 50)
    mlflow.log_param("alpha", 40)
    mlflow.log_param("n_users", n_users)
    mlflow.log_param("n_movies", n_items)
    mlflow.log_param("n_interactions", C.nnz)
    mlflow.log_param("sparsity_pct", round(C.nnz / (n_items * n_users) * 100, 4))

    # Metrics
    mlflow.log_metric("training_time_seconds", training_time)

    # Artifacts
    mlflow.log_artifact("/content/drive/MyDrive/als_model.pkl")
    mlflow.log_artifact("/content/drive/MyDrive/user_mapper.pkl")
    mlflow.log_artifact("/content/drive/MyDrive/user_inv_mapper.pkl")
    mlflow.log_artifact("/content/drive/MyDrive/movie_mapper.pkl")
    mlflow.log_artifact("/content/drive/MyDrive/movie_inv_mapper.pkl")
    mlflow.log_artifact("/content/drive/MyDrive/als_confidence_matrix.npz")

    # Tags
    mlflow.set_tag("model_type", "collaborative_filtering")
    mlflow.set_tag("algorithm", "ALS")
    mlflow.set_tag("framework", "implicit")
    mlflow.set_tag("status", "production_ready")

NameError: name 'datetime' is not defined

In [9]:
print("\n" + "=" * 60)
print("VALIDATION")
print("=" * 60)

# Check first few rows
print("\nSample of first 5 processed rows:")
print(interactions[['userId', 'tmdb_id', 'preference', 'confidence', 'user_idx', 'item_idx']].head(10))

# Check matrix statistics
print(f"\nFinal matrix shape: {C.shape}")
print(f"Total interactions: {C.nnz:,}")
print(f"Sparsity: {(1 - C.nnz / (n_items * n_users)) * 100:.4f}% empty")


VALIDATION

Sample of first 5 processed rows:
   userId  tmdb_id  preference  confidence  user_idx  item_idx
0   38150     1600           1        33.0     38149      1136
1   44717     8012           0        25.0     44716        20
2   44717      807           1        41.0     44716        46
3   44717      623           0        25.0     44716      1047
4   45491     8844           1        33.0     45490         1
5  126622      807           1        41.0    126615        46
6  126622      629           1        41.0    126615        49
7  126622    97406           1        33.0    126615        54
8  126622    11010           1        41.0    126615        57
9   45491    11860           1        41.0     45490         6

Final matrix shape: (45423, 270882)
Total interactions: 25,981,437
Sparsity: 99.7888% empty


In [10]:
# Test recommendations with correct model

# Load mappers
with open('/content/drive/MyDrive/user_mapper.pkl', 'rb') as f:
    user_mapper = pickle.load(f)

with open('/content/drive/MyDrive/movie_mapper.pkl', 'rb') as f:
    movie_mapper = pickle.load(f)

# Create inverse movie mapper
movie_inv_mapper = {idx: tmdb_id for tmdb_id, idx in movie_mapper.items()}

# Pick a user
sample_user_id = 38150
user_idx = user_mapper[sample_user_id]

# Get user's items from the training matrix
user_items = C_for_training[user_idx]  # Now this works correctly

# Get recommendations
recommendations = model.recommend(
    user_idx,
    user_items,
    N=10,
    filter_already_liked_items=True
)

print(f"\nTop 10 recommendations for user {sample_user_id}:")
for item_idx, score in zip(recommendations[0], recommendations[1]):
    tmdb_id = movie_inv_mapper[item_idx]
    print(f"  Movie {tmdb_id}: score {score:.4f}")

print(f"\nRaw recommendation indices: {recommendations[0]}")
print(f"Max index: {max(recommendations[0])} (should be <= 45422)")


Top 10 recommendations for user 38150:
  Movie 10451: score 1.2987
  Movie 10149: score 1.2379
  Movie 2759: score 1.2134
  Movie 11010: score 1.1810
  Movie 235: score 1.0673
  Movie 10997: score 1.0665
  Movie 11448: score 1.0310
  Movie 695: score 1.0299
  Movie 11382: score 1.0295
  Movie 9549: score 1.0123

Raw recommendation indices: [ 228  191  340   57 1214  441   51  530  343 1187]
Max index: 1214 (should be <= 45422)


In [11]:

import pickle

# Your mappers already exist from before:
with open('/content/drive/MyDrive/user_mapper.pkl', 'rb') as f:
    user_mapper = pickle.load(f)

with open('/content/drive/MyDrive/movie_mapper.pkl', 'rb') as f:
    movie_mapper = pickle.load(f)

# Create inverse mappers (if not already saved)
user_inv_mapper = {idx: uid for uid, idx in user_mapper.items()}
movie_inv_mapper = {idx: tmdb_id for tmdb_id, idx in movie_mapper.items()}

# SAVE THEM
with open('/content/drive/MyDrive/user_inv_mapper.pkl', 'wb') as f:
    pickle.dump(user_inv_mapper, f)
print("✓ Saved: user_inv_mapper.pkl")

with open('/content/drive/MyDrive/movie_inv_mapper.pkl', 'wb') as f:
    pickle.dump(movie_inv_mapper, f)
print("✓ Saved: movie_inv_mapper.pkl")

# Also save the training matrix for future use
from scipy.sparse import save_npz
save_npz('/content/drive/MyDrive/als_training_matrix.npz', C_for_training)
print("✓ Saved: als_training_matrix.npz")

✓ Saved: user_inv_mapper.pkl
✓ Saved: movie_inv_mapper.pkl
✓ Saved: als_training_matrix.npz


In [12]:
# Load movie metadata to see titles
movies_df = pd.read_csv('/content/drive/MyDrive/movies_merged.csv')

print("\nRecommended movies for user 38150:")
for idx in [228, 191, 340, 57, 1214, 441, 51, 530, 343, 1187]:
    tmdb_id = movie_inv_mapper[idx]
    movie = movies_df[movies_df['id'] == tmdb_id]
    if len(movie) > 0:
        title = movie['title'].iloc[0]
        year = movie['release_year'].iloc[0]
        print(f"  Index {idx}: Movie {tmdb_id} - {title} ({year})")


Recommended movies for user 38150:
  Index 228: Movie 10451 - Eat Drink Man Woman (1994.0)
  Index 191: Movie 10149 - Smoke (1995.0)
  Index 340: Movie 2759 - The Adventures of Priscilla, Queen of the Desert (1994.0)
  Index 57: Movie 11010 - The Postman (1994.0)
  Index 1214: Movie 235 - Stand by Me (1986.0)
  Index 441: Movie 10997 - Farewell My Concubine (1993.0)
  Index 51: Movie 11448 - Mighty Aphrodite (1995.0)
  Index 530: Movie 695 - Short Cuts (1993.0)
  Index 343: Movie 11382 - Bullets Over Broadway (1994.0)
  Index 1187: Movie 9549 - The Right Stuff (1983.0)


/tmp/ipython-input-32789/265558491.py:2: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_df = pd.read_csv('/content/drive/MyDrive/movies_merged.csv')


In [ ]:
movies_df.info()